# IGCD Change Map Set Viewer

Visualize one glacier sample as a matched set: baseline image (`t1`), target image (`t2`), corresponding 4-class change map, and optional change overlay on either image.

In [1]:
INSTALL_DEPENDENCIES = True

if INSTALL_DEPENDENCIES:
    %pip install -q rasterio matplotlib numpy pandas ipywidgets


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 45.8 MB/s eta 0:00:00


In [2]:
from pathlib import Path
import sys

try:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_ROOT = Path('/content/drive/MyDrive/IGCD')
except Exception:
    PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()

PROJECT_ROOT.mkdir(parents=True, exist_ok=True)
sys.path.insert(0, str(PROJECT_ROOT))
print({'project_root': str(PROJECT_ROOT)})


Mounted at /content/drive
{'project_root': '/content/drive/MyDrive/IGCD'}


In [3]:
import ipywidgets as widgets
import matplotlib.patches as mpatches
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import rasterio
from IPython.display import clear_output, display

from igcd.config import load_config
from igcd.raster_viewer import default_rgb_bands, open_raster_preview, rgb_composite
from igcd.visualization import (
    CHANGE_COLORS,
    CHANGE_LABELS,
    blend_change_overlay,
    change_class_counts,
    change_mask_to_rgb,
)


In [4]:
config = load_config(PROJECT_ROOT / 'config' / 'config.json')
BASE_YEAR = config.baseline_year
TARGET_YEAR = config.target_year
MAX_PREVIEW_SIZE = 1200
PREVIEW_DIR = config.paths['reports'] / 'change_set_previews'
PREVIEW_DIR.mkdir(parents=True, exist_ok=True)

print({'baseline_year': BASE_YEAR, 'target_year': TARGET_YEAR, 'preview_dir': str(PREVIEW_DIR)})


{'baseline_year': 2018, 'target_year': 2023, 'preview_dir': '/content/drive/MyDrive/IGCD/reports/change_set_previews'}


In [5]:
def discover_change_sets():
    rows = []
    change_root = config.paths['processed'] / 'change_masks'
    for change_path in sorted(change_root.glob('*_change.tif')):
        parts = change_path.stem.split('_')
        glacier_id = '_'.join(parts[:2])
        image_t1 = config.paths['exports'] / str(BASE_YEAR) / f'{glacier_id}_{BASE_YEAR}_sentinel.tif'
        image_t2 = config.paths['exports'] / str(TARGET_YEAR) / f'{glacier_id}_{TARGET_YEAR}_sentinel.tif'
        mask_t1 = config.paths['processed'] / 'masks' / str(BASE_YEAR) / f'{glacier_id}_{BASE_YEAR}_mask.tif'
        mask_t2 = config.paths['processed'] / 'masks' / str(TARGET_YEAR) / f'{glacier_id}_{TARGET_YEAR}_mask.tif'
        rows.append({
            'glacier_id': glacier_id,
            'image_t1': image_t1,
            'image_t2': image_t2,
            'mask_t1': mask_t1,
            'mask_t2': mask_t2,
            'change': change_path,
            'complete': image_t1.exists() and image_t2.exists() and change_path.exists(),
        })
    return pd.DataFrame(rows)

sets_df = discover_change_sets()
if sets_df.empty:
    raise FileNotFoundError(
        'No change maps found. Run RUN_SYNC_EXPORTS=True and RUN_CHANGE_MASKS=True first.'
    )

display(sets_df.head())
print({'change_sets': len(sets_df), 'complete_sets': int(sets_df['complete'].sum())})


,glacier_id,image_t1,image_t2,mask_t1,mask_t2,change,complete
0,IGCD_000001,/content/drive/MyDrive/IGCD/exports/2018/IGCD_...,/content/drive/MyDrive/IGCD/exports/2023/IGCD_...,/content/drive/MyDrive/IGCD/data/processed/mas...,/content/drive/MyDrive/IGCD/data/processed/mas...,/content/drive/MyDrive/IGCD/data/processed/cha...,False
1,IGCD_000002,/content/drive/MyDrive/IGCD/exports/2018/IGCD_...,/content/drive/MyDrive/IGCD/exports/2023/IGCD_...,/content/drive/MyDrive/IGCD/data/processed/mas...,/content/drive/MyDrive/IGCD/data/processed/mas...,/content/drive/MyDrive/IGCD/data/processed/cha...,True
2,IGCD_000003,/content/drive/MyDrive/IGCD/exports/2018/IGCD_...,/content/drive/MyDrive/IGCD/exports/2023/IGCD_...,/content/drive/MyDrive/IGCD/data/processed/mas...,/content/drive/MyDrive/IGCD/data/processed/mas...,/content/drive/MyDrive/IGCD/data/processed/cha...,True
3,IGCD_000006,/content/drive/MyDrive/IGCD/exports/2018/IGCD_...,/content/drive/MyDrive/IGCD/exports/2023/IGCD_...,/content/drive/MyDrive/IGCD/data/processed/mas...,/content/drive/MyDrive/IGCD/data/processed/mas...,/content/drive/MyDrive/IGCD/data/processed/cha...,True
4,IGCD_000007,/content/drive/MyDrive/IGCD/exports/2018/IGCD_...,/content/drive/MyDrive/IGCD/exports/2023/IGCD_...,/content/drive/MyDrive/IGCD/data/processed/mas...,/content/drive/MyDrive/IGCD/data/processed/mas...,/content/drive/MyDrive/IGCD/data/processed/cha...,True


{'change_sets': 876, 'complete_sets': 875}


In [6]:
def read_change_mask_preview(path, out_shape=None, max_size=1200):
    with rasterio.open(path) as src:
        if out_shape is None:
            scale = min(1.0, max_size / max(src.width, src.height))
            out_height = max(1, int(src.height * scale))
            out_width = max(1, int(src.width * scale))
        else:
            out_height, out_width = out_shape
        data = src.read(1, out_shape=(out_height, out_width), resampling=0)
    return data.astype('uint8')


def load_set(glacier_id):
    row = sets_df.loc[sets_df['glacier_id'] == glacier_id].iloc[0]
    t1 = open_raster_preview(row['image_t1'], max_size=MAX_PREVIEW_SIZE)
    t2 = open_raster_preview(row['image_t2'], max_size=MAX_PREVIEW_SIZE)
    out_shape = (t1.data.shape[1], t1.data.shape[2])
    change = read_change_mask_preview(row['change'], out_shape=out_shape)
    return row, t1, t2, change


In [8]:
from rasterio.enums import Resampling

def read_change_mask_preview(path, out_shape=None, max_size=1200):
    with rasterio.open(path) as src:
        if out_shape is None:
            scale = min(1.0, max_size / max(src.width, src.height))
            out_height = max(1, int(src.height * scale))
            out_width = max(1, int(src.width * scale))
        else:
            out_height, out_width = out_shape
        data = src.read(1, out_shape=(out_height, out_width), resampling=Resampling.nearest)
    return data.astype('uint8')

def load_set(glacier_id):
    row = sets_df.loc[sets_df['glacier_id'] == glacier_id].iloc[0]
    t1 = open_raster_preview(row['image_t1'], max_size=MAX_PREVIEW_SIZE)
    t2 = open_raster_preview(row['image_t2'], max_size=MAX_PREVIEW_SIZE)
    out_shape = (t1.data.shape[1], t1.data.shape[2])
    change = read_change_mask_preview(row['change'], out_shape=out_shape)
    return row, t1, t2, change


complete_ids = sets_df.loc[sets_df['complete'], 'glacier_id'].tolist()
if not complete_ids:
    raise FileNotFoundError('Change maps exist, but matching t1/t2 Sentinel files are missing.')

first_row, first_t1, first_t2, first_change = load_set(complete_ids[0])
default_r, default_g, default_b = default_rgb_bands(first_t1.band_count)
band_options = [(label, idx) for idx, label in enumerate(first_t1.band_labels, start=1)]

glacier = widgets.Dropdown(options=complete_ids, value=complete_ids[0], description='Glacier')
red = widgets.Dropdown(options=band_options, value=default_r, description='Red')
green = widgets.Dropdown(options=band_options, value=default_g, description='Green')
blue = widgets.Dropdown(options=band_options, value=default_b, description='Blue')
stretch = widgets.Dropdown(
    options=[('Percentile 2-98', 'percentile'), ('Min / max', 'minmax'), ('Mean +/- 2 std', 'stddev')],
    value='percentile',
    description='Stretch',
)
low = widgets.FloatSlider(value=2, min=0, max=20, step=0.5, description='Low %')
high = widgets.FloatSlider(value=98, min=80, max=100, step=0.5, description='High %')
gamma = widgets.FloatSlider(value=1.0, min=0.2, max=3.0, step=0.1, description='Gamma')
overlay_target = widgets.Dropdown(
    options=[('Target image', 't2'), ('Baseline image', 't1')],
    value='t2',
    description='Overlay on',
)
alpha = widgets.FloatSlider(value=0.45, min=0.05, max=0.95, step=0.05, description='Alpha')
show_overlay = widgets.Checkbox(value=True, description='Show overlay panel')
save_button = widgets.Button(description='Save PNG')
status = widgets.HTML(value='')
out = widgets.Output()
last_figure = {'fig': None}

In [9]:
def _legend_handles():
    handles = []
    for label, color in CHANGE_COLORS.items():
        rgb = tuple(channel / 255 for channel in color)
        handles.append(mpatches.Patch(color=rgb, label=f'{label}: {CHANGE_LABELS[label]}'))
    return handles


def _update_band_options(preview):
    options = [(label, idx) for idx, label in enumerate(preview.band_labels, start=1)]
    for dropdown in (red, green, blue):
        current = dropdown.value
        dropdown.options = options
        dropdown.value = current if current <= preview.band_count else 1


def render(*_):
    with out:
        clear_output(wait=True)
        row, t1, t2, change = load_set(glacier.value)
        _update_band_options(t1)
        t1_rgb = rgb_composite(
            t1, red.value, green.value, blue.value,
            mode=stretch.value,
            lower_percentile=low.value,
            upper_percentile=high.value,
            gamma=gamma.value,
        )
        t2_rgb = rgb_composite(
            t2, red.value, green.value, blue.value,
            mode=stretch.value,
            lower_percentile=low.value,
            upper_percentile=high.value,
            gamma=gamma.value,
        )
        change_rgb = change_mask_to_rgb(change)
        panels = 4 if show_overlay.value else 3
        fig, axes = plt.subplots(1, panels, figsize=(5.5 * panels, 6))
        axes[0].imshow(t1_rgb)
        axes[0].set_title(f'{glacier.value} - {BASE_YEAR}')
        axes[1].imshow(t2_rgb)
        axes[1].set_title(f'{glacier.value} - {TARGET_YEAR}')
        axes[2].imshow(change_rgb)
        axes[2].set_title('Change map')
        if show_overlay.value:
            base = t2_rgb if overlay_target.value == 't2' else t1_rgb
            overlay = blend_change_overlay(base, change, alpha=alpha.value)
            axes[3].imshow(overlay)
            axes[3].set_title(f'Change overlay on {overlay_target.value}')
        for ax in axes:
            ax.axis('off')
        fig.legend(
            handles=_legend_handles(),
            loc='lower center',
            ncol=4,
            bbox_to_anchor=(0.5, -0.03),
        )
        fig.tight_layout(rect=(0, 0.05, 1, 1))
        plt.show()
        counts = pd.DataFrame(
            [{'class': key, 'pixels': value} for key, value in change_class_counts(change).items()]
        )
        display(counts)
        last_figure['fig'] = fig


def save_png(_):
    if last_figure['fig'] is None:
        render()
    output = PREVIEW_DIR / f'{glacier.value}_{BASE_YEAR}_{TARGET_YEAR}_change_set.png'
    last_figure['fig'].savefig(output, dpi=180, bbox_inches='tight')
    status.value = f'<b>Saved:</b> {output}'

for widget in [glacier, red, green, blue, stretch, low, high, gamma, overlay_target, alpha, show_overlay]:
    widget.observe(render, names='value')
save_button.on_click(save_png)

controls = widgets.VBox([
    widgets.HBox([glacier, overlay_target, show_overlay, save_button]),
    widgets.HBox([red, green, blue]),
    widgets.HBox([stretch, low, high, gamma, alpha]),
    status,
])

display(controls, out)
render()


Output()

In [10]:
# Optional: inspect paths for the selected glacier.
selected = sets_df.loc[sets_df['glacier_id'] == glacier.value].T
selected.columns = ['value']
display(selected)


,value
glacier_id,IGCD_000002
image_t1,/content/drive/MyDrive/IGCD/exports/2018/IGCD_...
image_t2,/content/drive/MyDrive/IGCD/exports/2023/IGCD_...
mask_t1,/content/drive/MyDrive/IGCD/data/processed/mas...
mask_t2,/content/drive/MyDrive/IGCD/data/processed/mas...
change,/content/drive/MyDrive/IGCD/data/processed/cha...
complete,True
